In [1]:
import logging
import time
import os
import pickle

# from DataPrep import data_prep

import numpy as np
import matplotlib.pyplot as plt

#import tensorflow_datasets as tfds
import tensorflow as tf

# Import tf_text to load the ops used by the tokenizer saved model
#import tensorflow_text  # pylint: disable=unused-import
import pandas as pd
import numpy as np
import re
import seaborn as sns
import matplotlib as plt


from sklearn.model_selection import train_test_split


from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Model,  Sequential
from tensorflow.keras.layers import LSTM, GRU, Bidirectional, Dropout, Input, TimeDistributed, Dense, Activation, RepeatVector, Embedding, Concatenate
import tensorflow.keras.layers as layers
from tensorflow.keras.layers import Attention
from tensorflow.keras.optimizers import Adam, Adagrad
from keras.losses import sparse_categorical_crossentropy
logging.getLogger('tensorflow').setLevel(logging.ERROR)  # suppress warnings
import random

In [2]:
with open("Pichia_TrTs_2Target.pkl", "rb") as fp:
    Data_AllOrg = pickle.load(fp)
    
AA_tr = Data_AllOrg['AA_tr']
Cds_tr = Data_AllOrg['Cds_tr']

In [3]:
Settings = pd.read_csv('../BO_forHyperParameter/Arch1/Round1.csv').iloc[:, 1:]
Settings

,Enc hidden size,Enc Embedding size,Dec Embedding size,Dense Layer size,Dense Layer size aa,Drop rate,Drop rate aa
0,507.0,129.0,98.0,202.0,108.0,0.0,0.6
1,497.0,153.0,235.0,36.0,253.0,0.0,0.7
2,513.0,149.0,135.0,209.0,205.0,0.0,0.9


#### Network parameters

In [4]:
for i in np.arange(0, 3):
    Setting_no = i

    Max_length = 1000
    learning_rate = 0.001
    batch_size = 150
    epochs = 100
    aa_vocab_size = 25
    dna_vocab_size = 67


    hidden_size_enc = int(Settings['Enc hidden size'][Setting_no])
    hidden_size_enc_aa = int(Settings['Enc hidden size'][Setting_no])
    embedding_size_enc = int(Settings['Enc Embedding size'][Setting_no])
    embedding_size_dec = int(Settings['Dec Embedding size'][Setting_no])
    Dense_layer_size = int(Settings['Dense Layer size'][Setting_no])
    Dense_layer_size_aa = int(Settings['Dense Layer size aa'][Setting_no])

    drop_rate = Settings['Drop rate'][Setting_no]
    drop_rate_aa = Settings['Drop rate aa'][Setting_no]

    input_sequence = Input(shape=(Max_length,))
    encod_emb = Embedding(input_dim= aa_vocab_size, output_dim = embedding_size_enc,trainable=True, mask_zero = True)
    embedding = encod_emb(input_sequence)

    encoder = Bidirectional(GRU(hidden_size_enc, return_sequences=True, return_state = True),
                            merge_mode="concat", weights=None)

    encoder_sequence, encoder_final_f, encoder_final_b  = encoder(embedding)

    encoder_final = Concatenate(axis=-1)([encoder_final_f, encoder_final_b])


    
    decoder_inputs = Input(shape=(Max_length -1, ))
    decoder_inputs_aa = Input(shape=(Max_length, ))

    dex=  Embedding(input_dim = dna_vocab_size, output_dim = embedding_size_dec, trainable=True, mask_zero = True)


    final_dex= dex(decoder_inputs)
    final_dex_aa =  encod_emb(decoder_inputs_aa)


    decoder = GRU(2*hidden_size_enc, return_sequences = True, return_state = True)
    decoder_aa =  GRU(2*hidden_size_enc_aa, return_sequences = True, return_state = True)

    decoder_sequence, decoder_final = decoder(final_dex, initial_state=encoder_final)
    decoder_sequence_aa, decoder_final_aa = decoder_aa(final_dex_aa, initial_state=encoder_final)


    attn_layer = Attention()
    attn_out = attn_layer([decoder_sequence, encoder_sequence])
    attn_out_aa = attn_layer([decoder_sequence_aa, encoder_sequence])

    decoder_concat_input = Concatenate(axis=-1)([decoder_sequence, attn_out]) #decoder_sequence, 
    decoder_concat_input_aa = Concatenate(axis=-1)([decoder_sequence_aa, attn_out_aa]) #decoder_sequence,


    Intermediate_layer = TimeDistributed(Dense(Dense_layer_size, activation='tanh'))
    Intermediate_layer_aa= TimeDistributed(Dense(Dense_layer_size_aa, activation='tanh'))

    Intemediate_output = Intermediate_layer(decoder_concat_input) #decoder_concat_input
    Intemediate_output_aa = Intermediate_layer_aa(decoder_concat_input_aa) #decoder_concat_input


    dropout_layer = Dropout(drop_rate)
    dropout_output = dropout_layer(Intemediate_output)

    dropout_layer_aa = Dropout(drop_rate_aa)
    dropout_output_aa = dropout_layer(Intemediate_output_aa)

    dense_layer = TimeDistributed(Dense(dna_vocab_size, activation='softmax'))
    logits = dense_layer(dropout_output)

    dense_layer_aa = TimeDistributed(Dense(aa_vocab_size, activation='softmax'))
    logits_aa = dense_layer_aa(dropout_output_aa)

    enc_dec_model = Model([input_sequence, decoder_inputs, decoder_inputs_aa], [logits, logits_aa])

    enc_dec_model.compile(loss=sparse_categorical_crossentropy,
                  optimizer=Adam(learning_rate = learning_rate),
                  metrics=['accuracy'])
    enc_dec_model.summary()
    
    checkpoint_path = "/Desktop/training_2Target/Round1/" + str(Setting_no) + "/cp.ckpt"
    checkpoint_dir = os.path.dirname(checkpoint_path)

    # Create a callback that saves the model's weights
    cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath=checkpoint_path,
                                                 save_weights_only=True,
                                                 verbose=1)

    early_stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", min_delta=0, patience = 3,
        verbose=0, mode="auto", baseline=None, restore_best_weights=False)
    ## Train the model
    model_results = enc_dec_model.fit([AA_tr[:,1:Max_length+1], Cds_tr[:,0:Max_length-1], AA_tr[:,0:Max_length]], 
                                      [Cds_tr[:,1:Max_length],  AA_tr[:,1:Max_length+1]], 
                                      batch_size= batch_size, 
                                      epochs= epochs, 
                                  validation_split=0.2, callbacks=[cp_callback, early_stop])
    
    name_history = 'Loss_Evolution/Round1/Combo'+ str(Setting_no) + '.csv'
    pd.DataFrame(model_results.history).to_csv(name_history)
    
    name_model = 'saved_models/Round1/EncDec_' + str(Setting_no)
    enc_dec_model.save(name_model)

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 1000)]       0           []                               
                                                                                                  
 embedding (Embedding)          (None, 1000, 129)    3225        ['input_1[0][0]',                
                                                                  'input_3[0][0]']                
                                                                                                  
 input_2 (InputLayer)           [(None, 999)]        0           []                               
                                                                                                  
 bidirectional (Bidirectional)  [(None, 1000, 1014)  1940796     ['embedding[0][0]']          

Epoch 5/100
133/133 [==============================] - ETA: 0s - loss: 0.4491 - time_distributed_2_loss: 0.4484 - time_distributed_3_loss: 7.3382e-04 - time_distributed_2_accuracy: 0.4976 - time_distributed_3_accuracy: 0.9998
Epoch 5: saving model to /Desktop/training_2Target/Round1/0\cp.ckpt
133/133 [==============================] - 120s 902ms/step - loss: 0.4491 - time_distributed_2_loss: 0.4484 - time_distributed_3_loss: 7.3382e-04 - time_distributed_2_accuracy: 0.4976 - time_distributed_3_accuracy: 0.9998 - val_loss: 0.4499 - val_time_distributed_2_loss: 0.4492 - val_time_distributed_3_loss: 7.2716e-04 - val_time_distributed_2_accuracy: 0.4963 - val_time_distributed_3_accuracy: 0.9998
Epoch 6/100
133/133 [==============================] - ETA: 0s - loss: 0.4474 - time_distributed_2_loss: 0.4469 - time_distributed_3_loss: 5.6884e-04 - time_distributed_2_accuracy: 0.4998 - time_distributed_3_accuracy: 0.9998
Epoch 6: saving model to /Desktop/training_2Target/Round1/0\cp.ckpt
133/133

Epoch 17/100
133/133 [==============================] - ETA: 0s - loss: 0.3891 - time_distributed_2_loss: 0.3886 - time_distributed_3_loss: 4.5804e-04 - time_distributed_2_accuracy: 0.6021 - time_distributed_3_accuracy: 0.9998
Epoch 17: saving model to /Desktop/training_2Target/Round1/0\cp.ckpt
133/133 [==============================] - 120s 904ms/step - loss: 0.3891 - time_distributed_2_loss: 0.3886 - time_distributed_3_loss: 4.5804e-04 - time_distributed_2_accuracy: 0.6021 - time_distributed_3_accuracy: 0.9998 - val_loss: 0.4077 - val_time_distributed_2_loss: 0.4070 - val_time_distributed_3_loss: 6.1210e-04 - val_time_distributed_2_accuracy: 0.5755 - val_time_distributed_3_accuracy: 0.9998
Epoch 18/100
133/133 [==============================] - ETA: 0s - loss: 0.3675 - time_distributed_2_loss: 0.3670 - time_distributed_3_loss: 5.2771e-04 - time_distributed_2_accuracy: 0.6345 - time_distributed_3_accuracy: 0.9998
Epoch 18: saving model to /Desktop/training_2Target/Round1/0\cp.ckpt
133

Epoch 29/100
133/133 [==============================] - ETA: 0s - loss: 0.1281 - time_distributed_2_loss: 0.1276 - time_distributed_3_loss: 4.7749e-04 - time_distributed_2_accuracy: 0.9051 - time_distributed_3_accuracy: 0.9998
Epoch 29: saving model to /Desktop/training_2Target/Round1/0\cp.ckpt
133/133 [==============================] - 120s 901ms/step - loss: 0.1281 - time_distributed_2_loss: 0.1276 - time_distributed_3_loss: 4.7749e-04 - time_distributed_2_accuracy: 0.9051 - time_distributed_3_accuracy: 0.9998 - val_loss: 0.2467 - val_time_distributed_2_loss: 0.2461 - val_time_distributed_3_loss: 6.6596e-04 - val_time_distributed_2_accuracy: 0.8245 - val_time_distributed_3_accuracy: 0.9998
Epoch 30/100
133/133 [==============================] - ETA: 0s - loss: 0.1199 - time_distributed_2_loss: 0.1194 - time_distributed_3_loss: 5.0356e-04 - time_distributed_2_accuracy: 0.9115 - time_distributed_3_accuracy: 0.9998
Epoch 30: saving model to /Desktop/training_2Target/Round1/0\cp.ckpt
133

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_4 (InputLayer)           [(None, 1000)]       0           []                               
                                                                                                  
 embedding_2 (Embedding)        (None, 1000, 153)    3825        ['input_4[0][0]',                
                                                                  'input_6[0][0]']                
                                                                                                  
 input_5 (InputLayer)           [(None, 999)]        0           []                               
                                                                                                  
 bidirectional_1 (Bidirectional  [(None, 1000, 994),  1944264    ['embedding_2[0][0]']      

...

Epoch 41/100
133/133 [==============================] - ETA: 0s - loss: 0.1297 - time_distributed_6_loss: 0.1294 - time_distributed_7_loss: 2.9869e-04 - time_distributed_6_accuracy: 0.9020 - time_distributed_7_accuracy: 0.9999
Epoch 41: saving model to /Desktop/training_2Target/Round1/1\cp.ckpt
133/133 [==============================] - 120s 901ms/step - loss: 0.1297 - time_distributed_6_loss: 0.1294 - time_distributed_7_loss: 2.9869e-04 - time_distributed_6_accuracy: 0.9020 - time_distributed_7_accuracy: 0.9999 - val_loss: 0.2714 - val_time_distributed_6_loss: 0.2709 - val_time_distributed_7_loss: 4.8433e-04 - val_time_distributed_6_accuracy: 0.8137 - val_time_distributed_7_accuracy: 0.9998
Epoch 42/100
133/133 [==============================] - ETA: 0s - loss: 0.1267 - time_distributed_6_loss: 0.1264 - time_distributed_7_loss: 2.9000e-04 - time_distributed_6_accuracy: 0.9045 - time_distributed_7_accuracy: 0.9999
Epoch 42: saving model to /Desktop/training_2Target/Round1/1\cp.ckpt
133

Model: "model_2"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_7 (InputLayer)           [(None, 1000)]       0           []                               
                                                                                                  
 embedding_4 (Embedding)        (None, 1000, 149)    3725        ['input_7[0][0]',                
                                                                  'input_9[0][0]']                
                                                                                                  
 input_8 (InputLayer)           [(None, 999)]        0           []                               
                                                                                                  
 bidirectional_2 (Bidirectional  [(None, 1000, 1026)  2043792    ['embedding_4[0][0]']      

Epoch 5/100
133/133 [==============================] - ETA: 0s - loss: 0.4493 - time_distributed_10_loss: 0.4488 - time_distributed_11_loss: 5.0286e-04 - time_distributed_10_accuracy: 0.4970 - time_distributed_11_accuracy: 0.9998
Epoch 5: saving model to /Desktop/training_2Target/Round1/2\cp.ckpt
133/133 [==============================] - 130s 976ms/step - loss: 0.4493 - time_distributed_10_loss: 0.4488 - time_distributed_11_loss: 5.0286e-04 - time_distributed_10_accuracy: 0.4970 - time_distributed_11_accuracy: 0.9998 - val_loss: 0.4497 - val_time_distributed_10_loss: 0.4490 - val_time_distributed_11_loss: 6.4969e-04 - val_time_distributed_10_accuracy: 0.4966 - val_time_distributed_11_accuracy: 0.9998
Epoch 6/100
133/133 [==============================] - ETA: 0s - loss: 0.4475 - time_distributed_10_loss: 0.4471 - time_distributed_11_loss: 4.4410e-04 - time_distributed_10_accuracy: 0.4996 - time_distributed_11_accuracy: 0.9998
Epoch 6: saving model to /Desktop/training_2Target/Round1/2

Epoch 17/100
133/133 [==============================] - ETA: 0s - loss: 0.3947 - time_distributed_10_loss: 0.3941 - time_distributed_11_loss: 5.7025e-04 - time_distributed_10_accuracy: 0.5944 - time_distributed_11_accuracy: 0.9998
Epoch 17: saving model to /Desktop/training_2Target/Round1/2\cp.ckpt
133/133 [==============================] - 130s 978ms/step - loss: 0.3947 - time_distributed_10_loss: 0.3941 - time_distributed_11_loss: 5.7025e-04 - time_distributed_10_accuracy: 0.5944 - time_distributed_11_accuracy: 0.9998 - val_loss: 0.4113 - val_time_distributed_10_loss: 0.4107 - val_time_distributed_11_loss: 6.0063e-04 - val_time_distributed_10_accuracy: 0.5705 - val_time_distributed_11_accuracy: 0.9998
Epoch 18/100
133/133 [==============================] - ETA: 0s - loss: 0.3743 - time_distributed_10_loss: 0.3737 - time_distributed_11_loss: 5.1024e-04 - time_distributed_10_accuracy: 0.6256 - time_distributed_11_accuracy: 0.9998
Epoch 18: saving model to /Desktop/training_2Target/Roun

Epoch 29/100
133/133 [==============================] - ETA: 0s - loss: 0.1266 - time_distributed_10_loss: 0.1259 - time_distributed_11_loss: 7.9847e-04 - time_distributed_10_accuracy: 0.9068 - time_distributed_11_accuracy: 0.9997
Epoch 29: saving model to /Desktop/training_2Target/Round1/2\cp.ckpt
133/133 [==============================] - 130s 978ms/step - loss: 0.1266 - time_distributed_10_loss: 0.1259 - time_distributed_11_loss: 7.9847e-04 - time_distributed_10_accuracy: 0.9068 - time_distributed_11_accuracy: 0.9997 - val_loss: 0.2513 - val_time_distributed_10_loss: 0.2503 - val_time_distributed_11_loss: 9.5254e-04 - val_time_distributed_10_accuracy: 0.8203 - val_time_distributed_11_accuracy: 0.9997
Epoch 30/100
133/133 [==============================] - ETA: 0s - loss: 0.1182 - time_distributed_10_loss: 0.1176 - time_distributed_11_loss: 6.5780e-04 - time_distributed_10_accuracy: 0.9131 - time_distributed_11_accuracy: 0.9997
Epoch 30: saving model to /Desktop/training_2Target/Roun

#### Length and its plot

In [ ]:
length_aa = []
for i in range(len(AA_seq_tokenized)):
    length_aa.append(len(AA_seq_tokenized[i]))
print(max(length_aa))

sns.histplot(length_aa)


In [ ]:
 Cds_ts[0,0]